# ChEMBL activities for BRAF (UniProt entry name lookup)

`chem.chembl.download_activities` accepts a ChEMBL target id, a UniProt
accession, or a UniProt entry name (mnemonic) such as `BRAF_HUMAN` — it
resolves the id, calls the ChEMBL REST API directly (no
`chembl_webresource_client`), keeps only records with a pChEMBL value,
filters by molecular weight, and (when `normalize_smiles=True`) standardizes
each compound via the ChEMBL Structure Pipeline and aggregates duplicate
compounds into count/mean/median/std.

In [ ]:
from chem import chembl

n = chembl.download_activities(
    "BRAF_HUMAN",
    mw=[250, 650],
    normalize_smiles=True,
    output="braf_activities.tsv",
)
n

### Preview the result

In [ ]:
import pandas as pd

df = pd.read_csv("braf_activities.tsv", sep="\t")
print(df.size)
df.sort_values("pchembl_mean", ascending=False).head(10)

### Draw the top 10 most potent compounds

In [ ]:
from rdkit import Chem
from rdkit.Chem.Draw import MolsToGridImage

top10 = df.sort_values("pchembl_median", ascending=False).head(10)
mols = [Chem.MolFromSmiles(smi) for smi in top10["smiles"]]
legends = [
    f"{cid} pchembl_median={v}"
    for cid, v in zip(top10["parent_chembl_id"], top10["pchembl_median"])
]
MolsToGridImage(mols, molsPerRow=3, subImgSize=(350, 250), legends=legends)